<!-- notebook-header -->
# Pipelines de ML Automatizados

**Modulo:** 06 - MLOps  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** CI/CD, testes, orquestracao, feature stores, DVC e pipelines end-to-end.


# 6.4 Pipelines de ML Automatizados

Orquestracao, versionamento e MLOps em escala.


## Prerequisitos

Este notebook cobre automatizacao de pipelines ML.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import json, hashlib, time, logging
from typing import Tuple, Dict, List, Any

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('pipeline')

print('=== Pipelines de ML Automatizadas ===')


=== Pipelines de ML Automatizadas ===


## 1. Conceitos

**O que observar:**
Pipeline automatizado = sequencia de etapas sem intervencao humana.

**O que concluir:**
Automacao permite modelo sempre atualizado.

### Conexao com MLOps

MLOps e engenharia de pipelines ML.

**Por que em ML:**
Modelos precisam retreinar com novos dados.


In [2]:
class Stage:
    def __init__(self, name):
        self.name = name
        self.status = 'init'

class DataIngestion(Stage):
    def __init__(self, ns=100, nf=5):
        super().__init__('DataIngestion')
        self.ns = ns
        self.nf = nf
    def execute(self, src):
        logger.info(f'Ingestao: {src}')
        X = np.random.randn(self.ns, self.nf)
        y = (X[:, 0] + X[:, 1] > 0).astype(int)
        self.status = 'done'
        return X, y

ing = DataIngestion(100, 5)
X, y = ing.execute('data.csv')
print(f'Dados: X={X.shape}, y={y.shape}')


INFO:pipeline:Ingestao: data.csv


Dados: X=(100, 5), y=(100,)


## 2. Validacao

**O que observar:**
Validacao checa qualidade de dados antes de treinar.

**O que concluir:**
Dados ruins => modelo ruim. Validacao e primeiro filtro.

### Conexao com data quality

Data quality e prerequisito de ML bom.

**Por que em ML:**
Lixo entra => lixo sai. Modelo nao consegue consertar dados ruins.


In [3]:
class Validation(Stage):
    def execute(self, X, y):
        logger.info('Validando...')
        if X.ndim != 2: return False
        if X.shape[0] != y.shape[0]: return False
        if np.isnan(X).any(): return False
        logger.info('Validacao OK')
        self.status = 'done'
        return True

val = Validation('Validation')
ok = val.execute(X, y)
print(f'Validacao: {ok}')


INFO:pipeline:Validando...


INFO:pipeline:Validacao OK


Validacao: True


## 3. Feature Engineering

**O que observar:**
Transform dados brutos em features que modelo aprende.

**O que concluir:**
Boas features => modelo melhor. Feature engineering e criativo.

### Conexao com domain knowledge

Melhores features vem de entender problema.

**Por que em ML:**
Algoritmos nao descobrem todas patterns sozinhos.


In [4]:
class Features(Stage):
    def __init__(self):
        super().__init__('Features')
        self.mean = None
        self.std = None
    def fit(self, X):
        self.mean = X.mean(axis=0)
        self.std = X.std(axis=0) + 1e-8
    def execute(self, X):
        logger.info('Feature Engineering...')
        if self.mean is not None:
            X = (X - self.mean) / self.std
        self.status = 'done'
        return X

feat = Features()
feat.fit(X)
X = feat.execute(X)
print(f'Features normalized: mean={X.mean():.3f}, std={X.std():.3f}')


INFO:pipeline:Feature Engineering...


Features normalized: mean=0.000, std=1.000


## 4. Treinamento

**O que observar:**
Treino encontra pesos que minimizam erro.

**O que concluir:**
Em pipeline, treino e repetido com dados novos periodicamente.

### Conexao com reproducibilidade

Seed fixo => mesmo resultado toda vez.

**Por que em ML:**
Retreinamento faz modelo se adaptar a patterns novos.


In [5]:
class LogisticReg:
    def __init__(self, lr=0.01, ep=50):
        self.lr = lr
        self.ep = ep
        self.w = None
        self.b = None
    def fit(self, X, y):
        n, m = X.shape
        self.w = np.zeros(m)
        self.b = 0
        for i in range(self.ep):
            z = X @ self.w + self.b
            p = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            dw = X.T @ (p - y) / n
            db = (p - y).mean()
            self.w -= self.lr * dw
            self.b -= self.lr * db
    def predict(self, X):
        z = X @ self.w + self.b
        return (1 / (1 + np.exp(-np.clip(z, -500, 500))) >= 0.5).astype(int)

np.random.seed(42)
Xt = np.random.randn(100, 5)
yt = (Xt[:, 0] + Xt[:, 1] > 0).astype(int)
m = LogisticReg(lr=0.1, ep=50)
m.fit(Xt, yt)
print('Modelo treinado')


Modelo treinado


## 5. Avaliacao

**O que observar:**
Avaliacao determina se modelo e bom bastante.

**O que concluir:**
Avaliacao e quality gate. Novo modelo so deploy se passou.

### Conexao com decisoes

Sem avaliacao rigorosa, modelos ruins chegam producao.

**Por que em ML:**
Nao tem como saber se modelo bom sem testar.


In [6]:
class Eval(Stage):
    def __init__(self, min_acc=0.65):
        super().__init__('Eval')
        self.min_acc = min_acc
        self.metrics = {}
    def exec(self, m, Xt, yt):
        yp = m.predict(Xt)
        acc = (yp == yt).mean()
        self.metrics = {'acc': acc}
        logger.info(f'Accuracy: {acc:.3f}')
        ok = acc >= self.min_acc
        self.status = 'done' if ok else 'fail'
        return ok

Xtest = np.random.randn(30, 5)
ytest = (Xtest[:, 0] > 0).astype(int)
ev = Eval(min_acc=0.5)
passed = ev.exec(m, Xtest, ytest)
print(f'Passou: {passed}')


INFO:pipeline:Accuracy: 0.733


Passou: True


## 6. DAG

**O que observar:**
DAG define ordem de execucao de etapas.

**O que concluir:**
DAG garante dependencias corretas sao respeitadas.

### Conexao com orquestracao

Orquestracao e coordenacao de etapas. DAG deixa explicito.

**Por que em ML:**
Workflows ML sao complexos. DAG organiza dependencias.


In [7]:
class DAG:
    def __init__(self, name):
        self.name = name
        self.stages = {}
        self.deps = {}
    def add(self, sid, stage):
        self.stages[sid] = stage
        self.deps[sid] = []
    def depend(self, down, up):
        self.deps[down].append(up)
    def topo(self):
        vis = set()
        ord = []
        def dfs(s):
            if s in vis: return
            vis.add(s)
            for p in self.deps[s]:
                dfs(p)
            ord.append(s)
        for s in self.stages:
            dfs(s)
        return ord

dag = DAG('pipeline_v1')
dag.add('ing', DataIngestion(100, 5))
dag.add('val', Validation('V'))
dag.add('feat', Features())
dag.depend('val', 'ing')
dag.depend('feat', 'val')
print(f'Topo: {dag.topo()}')


Topo: ['ing', 'val', 'feat']


## 7. Error Handling

**O que observar:**
Em producao, coisas dao errado: conexoes falham, dados atrasam.

**O que concluir:**
Pipeline robusta tem retry, timeout, fallback, alertas.

### Conexao com resilience

Resiliencia = recuperar de falhas. Essencial em ML scale.

**Por que em ML:**
Novos dados podem ter distribuicoes diferentes. Modelo pode falhar.


In [8]:
class Retry:
    def __init__(self, max_t=3):
        self.max_t = max_t
    def exec(self, func, *args):
        for i in range(self.max_t):
            try:
                logger.info(f'Tentativa {i+1}/{self.max_t}')
                return func(*args)
            except Exception as e:
                logger.warning(f'Erro: {e}')
                if i == self.max_t - 1:
                    raise
                time.sleep(0.1)

def flaky_fn(x):
    if np.random.random() < 0.5:
        raise ValueError('Flaky')
    return x * 2

ret = Retry(max_t=3)
result = ret.exec(flaky_fn, 5)
print(f'Result: {result}')


INFO:pipeline:Tentativa 1/3


Result: 10


## 8. Model Registry

**O que observar:**
Registry rastreia versoes de modelos, metricas, timestamps.

**O que concluir:**
Versionamento permite rollback rapido se modelo novo piora.

### Conexao com auditoria

Registry e papel de auditoria. Responde: qual modelo rodou quando?

**Por que em ML:**
Modelos evoluem. Sem versioning, impossivel saber qual em producao.


In [9]:
class Registry:
    def __init__(self):
        self.models = {}
        self.prod = None
    def register(self, m, met):
        v = f'v{len(self.models)+1}'
        self.models[v] = {'m': m, 'met': met, 'ts': datetime.now().isoformat()}
        logger.info(f'Registered: {v}')
        return v
    def promote(self, v, min_a=0.65):
        if self.models[v]['met']['acc'] < min_a:
            logger.error('Nao passou')
            return False
        self.prod = v
        logger.info(f'Promoted: {v}')
        return True

reg = Registry()
v1 = reg.register(m, {'acc': 0.72})
reg.promote(v1, min_a=0.65)
print(f'Producao: {reg.prod}')


INFO:pipeline:Registered: v1


INFO:pipeline:Promoted: v1


Producao: v1


## 9. Monitoring

**O que observar:**
Monitoring acompanha performance de modelo em producao.

**O que concluir:**
Se performance cai, pipeline detecta e pode retreinar.

### Conexao com observability

Observability = saber estado do sistema. Crucial em producao.

**Por que em ML:**
Modelos degradam com tempo. Sem monitoring, nao percebe.


In [10]:
class Monitor:
    def __init__(self, base_acc=0.7):
        self.base_acc = base_acc
        self.hist = []
    def record(self, yt, yp):
        acc = (yp == yt).mean()
        deg = self.base_acc - acc
        if deg > 0.05:
            logger.warning(f'Degradacao: {deg:.3f}')
        self.hist.append({'acc': acc})
        return acc

mon = Monitor(base_acc=0.7)
yt1 = np.array([0, 1, 1, 0, 1])
yp1 = np.array([0, 1, 1, 0, 1])
a1 = mon.record(yt1, yp1)
yp2 = np.array([1, 0, 0, 1, 0])
a2 = mon.record(yt1, yp2)
print(f'Acuracias: {[a1, a2]}')


Acuracias: [np.float64(1.0), np.float64(0.0)]


## 10. Erros Comuns

### Erro: Pipeline sem Validacao

Problema: Treina com dados ruins.
Solucao: Sempre valide antes.

### Erro: Preprocessamento Inconsistente

Problema: Treino bom, producao ruim.
Solucao: Reutilize fit() de treino.

### Erro: Dados Vencidos

Problema: Performance piora com tempo.
Solucao: Retreine periodicamente.

### Erro: Sem Versionamento

Problema: Nao consegue voltar modelo bom.
Solucao: Use registry.

### Erro: Threshold Frouxo

Problema: Deploya modelos ruins.
Solucao: Set thresholds altos.


## 11. Exercicios

### TAREFA DO ALUNO 1: Retraining Scheduler

Implementar classe que verifica se e hora de retreinar.

### TAREFA DO ALUNO 2: Canary Deployment

Testar novo modelo em 10% antes de deploy total.

### TAREFA DO ALUNO 3: Feature Importance

Calcular quais features sao mais importantes.

## Solucoes

### SOLUCAO 1: Scheduler


### TAREFA DO ALUNO Exercicio 1: Retraining Scheduler

Implementar classe que verifica se hora de retreinar.


In [ ]:
# TAREFA DO ALUNO: Implementar scheduler simples
result = None  # Preencher
print(f'Exercicio 1: {result}')

In [12]:
# SOLUCAO - Exercicio 1
# Retraining Scheduler simples

import numpy as np
from datetime import datetime, timedelta

class RetrainingScheduler:
    def __init__(self, days=7):
        self.days = days
        self.last_training = datetime.now()
    
    def should_retrain(self, current_time=None):
        if current_time is None:
            current_time = datetime.now()
        elapsed = (current_time - self.last_training).days
        return elapsed >= self.days
    
    def record_training(self, time=None):
        self.last_training = time if time else datetime.now()

# Test
sched = RetrainingScheduler(days=7)
print(f'Should retrain now: {sched.should_retrain()}')
future = datetime.now() + timedelta(days=8)
print(f'Should retrain in 8 days: {sched.should_retrain(future)}')


Should retrain now: False
Should retrain in 8 days: True


In [ ]:
# TAREFA DO ALUNO: Implementar Canary deployment (testar em 10% antes de deploy total)
result = None  # Preencher
print(f'Exercicio 2: {result}')

In [14]:
# SOLUCAO - Exercicio 2
# Canary Deployment simples

import numpy as np

class SimpleModel:
    def predict(self, X):
        return (X[:, 0] > 0).astype(int)

class CanaryDeployment:
    def __init__(self, canary_fraction=0.1):
        self.canary_fraction = canary_fraction
        self.current_model = None
    
    def test_new_model(self, new_model, X, y):
        n_canary = max(1, int(len(X) * self.canary_fraction))
        idx_canary = np.random.choice(len(X), n_canary, replace=False)
        X_canary = X[idx_canary]
        y_canary = y[idx_canary]
        
        predictions = new_model.predict(X_canary)
        accuracy = np.mean(predictions == y_canary)
        
        if self.current_model is None or accuracy >= 0.7:
            self.current_model = new_model
            return True
        return False

# Test
X_test = np.random.randn(100, 3)
y_test = (X_test[:, 0] > 0).astype(int)

canary = CanaryDeployment(canary_fraction=0.1)
model = SimpleModel()
deployed = canary.test_new_model(model, X_test, y_test)
print(f'Model deployed: {deployed}')


Model deployed: True


In [ ]:
# TAREFA DO ALUNO: Calcular importancia de features
result = None  # Preencher
print(f'Exercicio 3: {result}')

In [16]:
# SOLUCAO - Exercicio 3
# Feature Importance de modelo

import numpy as np

class SimpleLinearModel:
    def __init__(self, n_features=5):
        self.weights = np.random.randn(n_features)
    
    def get_feature_importance(self):
        importance = np.abs(self.weights)
        importance = importance / importance.sum()
        return dict(zip([f'feature_{i}' for i in range(len(self.weights))], importance))

# Test
model = SimpleLinearModel(n_features=5)
importance = model.get_feature_importance()
for feat, imp in sorted(importance.items(), key=lambda x: x[1], reverse=True):
    print(f'{feat}: {imp:.4f}')


feature_3: 0.4419
feature_1: 0.3187
feature_4: 0.1346
feature_2: 0.0664
feature_0: 0.0384


In [17]:
class Scheduler:
    def __init__(self, days=7):
        self.days = days
        self.last = datetime.now()
    def should_retrain(self):
        elapsed = (datetime.now() - self.last).days
        return elapsed >= self.days
    def retrain(self, X, y):
        if not self.should_retrain():
            return None
        logger.info('Retraining...')
        m2 = LogisticReg()
        m2.fit(X, y)
        self.last = datetime.now()
        return m2

sched = Scheduler(days=7)
print(f'Retreinar? {sched.should_retrain()}')


Retreinar? False


### SOLUCAO 2: Canary


In [18]:
class Canary:
    def __init__(self, frac=0.1):
        self.frac = frac
        self.prod = None
    def test(self, new, Xt, yt):
        idx = np.random.choice(len(Xt), int(len(Xt)*self.frac))
        Xc = Xt[idx]
        yc = yt[idx]
        new_acc = (new.predict(Xc) == yc).mean()
        if self.prod:
            old_acc = (self.prod.predict(Xc) == yc).mean()
            if new_acc >= old_acc - 0.02:
                self.prod = new
                return True
            return False
        self.prod = new
        return True

can = Canary(frac=0.2)
m3 = LogisticReg()
m3.fit(Xt, yt)
ok = can.test(m3, Xtest, ytest)
print(f'Canary OK: {ok}')


Canary OK: True


### SOLUCAO 3: Feature Importance


In [19]:
def feat_imp(m, names=None):
    if m.w is None:
        raise ValueError('Not fitted')
    n = len(m.w)
    if names is None:
        names = [f'f{i}' for i in range(n)]
    imp = np.abs(m.w)
    imp = imp / imp.sum()
    return dict(zip(names, imp))

names = ['f0', 'f1', 'f2', 'f3', 'f4']
imp_dict = feat_imp(m, names)
print('Importance:', {k: f'{v:.3f}' for k, v in imp_dict.items()})


Importance: {'f0': '0.387', 'f1': '0.436', 'f2': '0.097', 'f3': '0.039', 'f4': '0.041'}


## 12. Conceitos Avancados

### O que observar sobre Pipeline Stages

Cada stage em uma pipeline deve ser independente e reutilizavel.
Stages compostas formam DAGs (Directed Acyclic Graphs) que definem fluxo de dados.

### O que concluir sobre Composicao

Separacao de concerns permite que cada etapa seja testada isoladamente.
Uma pipeline e composicao de stages que passam dados entre si.

### Conexao com arquitetura de software

Pipelines seguem mesmo principo de composition que padroes de design.
Cada stage tem responsabilidade unica e bem definida.

**Por que em ML:**
Modularidade permite reutilizar stages em diferentes pipelines.
Uma pipeline pode ser construida combinando stages existentes.


## 13. Validacao em Profundidade

### O que observar sobre qualidade de dados

Validacao verifica: shape, tipos, NaN, Inf, balance de classes, ranges esperados.
Cada cheque e uma barreira contra problemas de dados.

### O que concluir sobre preprocessing

Dados sujos podem ser detectados cedo na pipeline, antes de desperdicio.
Validacao precoce economiza tempo e recursos computacionais.

### Conexao com Assurance

Data quality assurance e similar a testing em software engineering.
Garantir dados bons antes de treinar modelo.

**Por que em ML:**
Algoritmos aprendem patterns nos dados. Se dados sao ruins, patterns sao ruins.


## 14. Orquestracao em Detalhe

### O que observar sobre DAGs

DAG permite visualizar pipeline como grafo.
Nodes = stages, Edges = dependencias de dados.

### O que concluir sobre paralelismo

Stages sem dependencias mutuas podem rodar em paralelo.
DAG permite identificar oportunidades de paralelizacao.

### Conexao com sistemas distribuidos

Orquestracao em larga escala requer frameworks como Airflow ou Prefect.
DAG e abstraccao que permite coordenar computacoes distribuidas.

**Por que em ML:**
Pipelines ML em producao sao complexas e envolvem multiplos stages.


## Resumo

### Hierarquia

```
MLOps
 
├─ Pipeline
 
│  
├─ Ingestao
 
│  
├─ Validacao
 
│  
├─ Features
 
│  
├─ Treino
 
│  
├─ Avaliacao
 
│  
└─ Deploy
 
├─ DAG
 
├─ Registry
 
├─ Monitor
 
└─ Error Handling
```

### Checklist

- [ ] Validacao de dados
- [ ] Features reproducivel
- [ ] Seed fixo
- [ ] Quality gates
- [ ] Novo > antigo
- [ ] Versioning
- [ ] Error handling
- [ ] Monitoring
- [ ] Documentation
- [ ] Logging
- [ ] Data versioning
- [ ] Alertas

### Proximos Passos

1. Scheduler
2. Canary
3. Feature store
4. Data drift
5. A/B testing
6. AutoML
7. Infrastructure
8. Multi-model